In [1]:
%pip install peft==0.4.0
!pip install datasets


  Using cached peft-0.4.0-py3-none-any.whl.metadata (21 kB)
Using cached peft-0.4.0-py3-none-any.whl (72 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.19.0
    Uninstalling peft-0.19.0:
      Successfully uninstalled peft-0.19.0


In [5]:
!pip install -q --upgrade torchao peft transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.7 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset("Abirate/english_quotes", split="train").shuffle(seed=42).select(range(len(load_dataset("Abirate/english_quotes", split="train")) // 10))
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)
train_sample = data.select(range(5))
display(train_sample)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})

In [6]:
import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],  # couches d'attention de BLOOM
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176
None


In [7]:

# Fill out the `Trainer` class.

import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
trainer.train()

# Load the PEFT model using pre-defined LoRA configs and foundation model. We set `is_trainable=False` to avoid further training.


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.0, metrics={'train_runtime': 955.3971, 'train_samples_per_second': 0.005, 'train_steps_per_second': 0.001, 'total_flos': 326604718080.0, 'train_loss': 0.0, 'epoch': 1.0})

In [10]:






import time

time_now = time.strftime("%Y%m%d_%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# Generate output tokens

inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=False,  # greedy decoding, plus stable
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

['Two things are infinite: ']
